In [1]:
import sys
sys.path.insert(0,'/nfs/nfs2/users/riadoshi/bigvision_palivla/')
sys.path.insert(0, '/nfs/nfs2/users/riadoshi/bigvision_palivla/src/')
sys.path.insert(0,'/nfs/nfs2/users/riadoshi/bigvision_palivla/dlimp')
sys.path.insert(0,'/nfs/nfs2/users/riadoshi/bigvision_palivla/models')

import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

import jax
import jax.experimental.multihost_utils as mhu
from jax.experimental import multihost_utils

from absl import flags
import optax
import numpy as np
import orbax.checkpoint as ocp
from transformers import AutoTokenizer
import mediapy
from ml_collections import config_flags
from typing import Any

from scalax.sharding import MeshShardingHelper, FSDPShardingRule

from palivla.components.train_state import ShardingMetadata
from palivla.model_components import ModelComponents
from octo.data.utils.data_utils import NormalizationType

from palivla.dataset import make_trajectory_dataset
import palivla
from palivla.components.sequence_builder import SequenceBuilder

from scripts.train import create_model

from big_vision.utils import Registry

from octo.data.oxe import make_oxe_dataset_kwargs
from octo.data.dataset import make_single_dataset

import importlib

2025-04-28 22:26:11.060379: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-28 22:26:11.064138: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-28 22:26:11.075852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745879171.095378 1679319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745879171.101178 1679319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745879171.115315 1679319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# Reset flags if the cell is re-run
try:
    flags.FLAGS.unparse_flags()
except AttributeError:
    flags.FLAGS._flag_values_dict.clear()

# Define the config flag only if it hasn't been defined
if "config" not in flags.FLAGS:
    config_flags.DEFINE_config_file(
        "config", "/nfs/nfs2/users/riadoshi/bigvision_palivla/configs/reasonings_only/all_reasonings_only.py", "Path to the config file."
    )

import sys
flags.FLAGS(sys.argv, known_only=True)

config = flags.FLAGS.config

In [3]:
# MODEL_PATH, MODEL_STEP = 'gs://kyle-checkpoints-c2/paligemma-checkpoints/solar-feather-24', 40000
# MODEL_PATH, MODEL_STEP = 'gs://multi-robot-bucket3/runs/vla/cot_all_07042025_075319', 140000
MODEL_PATH, MODEL_STEP = "gs://multi-robot-bucket3/runs/vla/constant_lr_reasonings_only_25042025_025425", 30000

mesh = MeshShardingHelper([-1], ["fsdp"])
sharding_metadata = ShardingMetadata(
    mesh=mesh,
    model_sharding_rule=FSDPShardingRule(
        "fsdp", fsdp_axis_size=mesh.mesh.shape["fsdp"]
    ),
)

optimizer = optax.identity()
print("Loading model params...")

model = create_model(config, sharding_metadata)
for load_fn, load_fn_kwargs in config.load_fns:
    load_fn = Registry.lookup(load_fn)
    load_fn(model, **load_fn_kwargs)

restore_manager = ocp.CheckpointManager(
    MODEL_PATH, options=ocp.CheckpointManagerOptions()
)
model.load_state(MODEL_STEP, restore_manager)
train_state = model.train_state

Loading model params...


Some kwargs in processor config are unused and will not have any effect: scale, time_horizon, min_token, action_dim, vocab_size. 


Replacing param /llm/embedder/input_embedding with subarray strategy


I0428 22:28:26.592797 1681434 google_auth_provider.cc:181] Running on GCE, using service account 180902422847-compute@developer.gserviceaccount.com


In [17]:
# viz_kwargs = config.visualization_datasets['droid_dataset'].to_dict()
# viz_kwargs['data_dir'] = 'gs://rail-orca-central2/resize_256_256/'
# viz_kwargs['use_cot'] = False
# viz_kwargs['cot_data_path'] = 'gs://multi-robot-bucket2/data/generated_data'
viz_kwargs = {'action_proprio_normalization_type': NormalizationType.NORMAL,
 'cot_data_path': None,
 'data_dir': 'gs://rail-orca-central2/resize_256_256/',
 'force_recompute_dataset_statistics': False,
 'frame_transform_kwargs': {'image_augment_kwargs': {},
  'resize_size': {'primary': [224, 224]}},
 'load_camera_views': ['primary'],
 'load_depth': False,
 'load_language': True,
 'load_proprio': True,
 'name': 'droid_dataset',
 'use_cot': False}
viz_dataset =  make_trajectory_dataset(**viz_kwargs,train=True)
viz_dataset_iterator = viz_dataset.iterator()



In [18]:
viz_trajectories = [traj for traj in viz_dataset_iterator]
# for t in viz_trajectories:
#     t['observation']['image_primary'] = t['observation']['image_high']
#     t['observation']['pad_mask_dict']['image_primary'] = t['observation']['pad_mask_dict']['image_high']
#     t['observation'].pop('image_high')
#     t['observation'].pop('proprio_bimanual')

2025-04-28 22:41:25.677197: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [19]:
from palivla.visualizations.chain_of_thought import parse_cot_string, visualize_trajectory, TrajectoryData
import matplotlib.pyplot as plt
import io
from PIL import Image


def create_viz(
    image: np.ndarray,
    pred_trajectory: TrajectoryData,
    target_trajectory: TrajectoryData,
    prompt_str: str,
) -> np.ndarray:
    """Create a side-by-side visualization comparing predicted and target trajectories.

    Args:
        image: Input image to show in both panels
        pred_trajectory: Predicted trajectory from the model
        target_trajectory: Target/ground truth trajectory
        prompt_str: Text prompt to show as subtitle

    Returns:
        PIL Image containing the rendered figure
    """
    # Create figure with two subplots side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Remove <pad> from prompt string
    prompt_str = prompt_str.replace("<pad>", "")
    fig.suptitle(prompt_str, wrap=True)

    # Plot predicted and target trajectories
    visualize_trajectory(ax1, image, pred_trajectory, "Predicted")
    visualize_trajectory(ax2, image, target_trajectory, "Target")

    # Adjust layout and convert to image
    plt.tight_layout()

    # Convert figure to PIL Image
    buf = io.BytesIO()
    plt.savefig(buf, format="png")
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)

    return img

def chain_of_thought(model, trajectory, idx):
    frame = jax.tree_map(lambda x: x[idx:idx+1], trajectory)
    frame["observation"] = jax.tree_map(lambda x: x[None], frame["observation"])
    frame["action"] = trajectory["action"][None, :1, :]

    if "reasonings" not in frame:
        frame["reasonings"] = np.array(["" for _ in range(len(frame["action"]))])

    # Predict chain-of-thought
    sequences = model.build_sequence(frame, begin_is_prompt=True)
    # print("sequence: ", sequences['gen'])
    viz_batch, sequences = mhu.broadcast_one_to_all(
        ({"observation": frame["observation"]}, sequences)
    )
    predicted_tokens = model.predict_tokens(
        viz_batch, sequences, use_ema_params=False, replicate=True
    )

    # Decode the tokens
    predicted_text_tokens = np.array(
        [model.language_tokenizer.decode(tok) for tok in predicted_tokens[0]]
    )

    target_gen_tokens = sequences["gen"]["tokens"]
    target_gen_text_tokens = np.array(
        [model.language_tokenizer.decode(tok) for tok in target_gen_tokens[0]]
    )

    prompt_tokens = sequences["prompt"]["tokens"]
    prompt_text_tokens = np.array(
        [
            model.language_tokenizer.decode(tok)
            for tok in prompt_tokens[0]
            if tok != "<pad>"
        ]
    )
    # print(prompt_text_tokens)

    # Get action start indices
    pred_action_start_idxs = (
        np.argwhere(predicted_text_tokens == "<begin_of_action>") + 1
    )
    pred_action_start_idx = (
        np.min(pred_action_start_idxs) if pred_action_start_idxs.size > 0 else -1
    )
    target_action_start_idx = (
        np.min(np.argwhere(target_gen_text_tokens == "<begin_of_action>")) + 1
    )

    # print("gt: ", sequences['gen']['tokens'])
    # print("pred: ", predicted_text_tokens[pred_action_start_idx:])

    # if 'palivla.components.sequence_builder' in sys.modules:
    #     del sys.modules['palivla.components.sequence_builder']
    # importlib.reload(sequence_builder)

    sequence_builder = SequenceBuilder(
        prompt_pad_length=50, 
        gen_pad_length=150,
        action_chunk_pad_length=4
    )

    actions, actions_mask = sequence_builder.batch_get_actions(
            predicted_tokens,
            model.language_tokenizer,
            model.action_tokenizer,
            begin_is_prompt=True,
            action_dim=14,
            action_chunk_sizes=[4],
    )
    
    # Concatenate tokens into strings
    prompt_str = "".join(prompt_text_tokens)
    pred_cot_str = "".join(predicted_text_tokens[:pred_action_start_idx])
    target_cot_str = "".join(target_gen_text_tokens[:target_action_start_idx])

    # Parse both predicted chain of thought string
    pred_trajectory = parse_cot_string(pred_cot_str)
    target_trajectory = parse_cot_string(target_cot_str)

    # Create side-by-side visualization
    image = frame["observation"]['image_primary'].squeeze()
    comparison_image = create_viz(
        image, pred_trajectory, target_trajectory, prompt_str
    ) # this is just gonna be predicted trajectory twice

    return comparison_image, actions



In [ ]:
all_actions = []

for _ in range(2):
    num = np.random.randint(0, 100)
    print(num)

    traj = viz_trajectories[num].copy()
    # while "banana" not in traj['task']['language_instruction'][0].decode('utf-8'):
    #     num = np.random.randint(0, 10)
    #     traj = viz_trajectories[num].copy()
    
    # traj['reasonings'] = np.array(["" for _ in range(len(traj['action']))])

    print(traj['observation'].keys())
    imgs = []
    for i in range(len(traj['action'])):
        if i%10 == 0:
            img, action = chain_of_thought(model, traj, i)
            # print(action)
            # all_actions.append(action)
            imgs.append(img)
    new_imgs = [np.array(img)[:, :, :3] for img in imgs]
    mediapy.show_video(new_imgs, fps=2)

In [2]:
all_actions = []

for _ in range(1):
    num = np.random.randint(0, 25)
    print(num)

    traj = viz_trajectories[num]
    # traj['reasonings'] = np.array(["" for _ in range(len(traj['action']))])

    print(traj['observation'].keys())

    img, action = chain_of_thought(model, traj)
    print(action)
    all_actions.append(action)
    mediapy.show_image(img)

NameError: name 'np' is not defined

In [ ]:
all_actions = np.array(all_actions).squeeze()
# plot the distribution for each action dim 

import matplotlib.pyplot as plt 
for i in range(6):
    plt.hist(all_actions[:, i], bins=10) # data is your numerical data, bins is the number of intervals
    plt.title(f"NORMALIZED: dim {i}")
    plt.show()

In [ ]:
DATASET_MEAN = [0.000217586566577665, 0.000125082762679085, -0.000171084306202828, -0.000161708827363327, -0.000252487370744347, 0.00025158081552945, 0.587948560714722]
DATASET_STD = [0.00963237509131432, 0.013500677421689, 0.0125105613842607, 0.0281452126801014, 0.0302824676036835, 0.0758553743362427, 0.487718880176544]

for i in range(6):
    plt.hist(DATASET_MEAN[i] + DATASET_STD[i] * all_actions[:, i], bins=20) # data is your numerical data, bins is the number of intervals
    plt.title(f"UNNORMALIZED: dim {i}")
    plt.show()

In [1]:
from octo.data.oxe import make_oxe_dataset_kwargs
from octo.data.dataset import make_single_dataset

dataset_name = "aloha_pen_uncap_diverse_dataset"
dataset_kwargs = make_oxe_dataset_kwargs(dataset_name,"gs://rail-orca-central2/resize_256_256")
dataset = make_single_dataset(
                                dataset_kwargs, 
                                frame_transform_kwargs=dict(
                                    resize_size={"primary": (256, 256)},
                                ),
                                train=True)
iterator = dataset.iterator()
num_trajs = len([1 for traj in iterator])

2025-03-20 22:54:51.652768: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-20 22:54:51.656043: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-20 22:54:51.666691: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742511291.684981  611338 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742511291.690488  611338 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742511291.704043  611338 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

AttributeError: module 'ml_dtypes' has no attribute 'float4_e2m1fn'
Cause: Unable to locate the source code of <function _gcd_import at 0x7fedf7583d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7fedf7583d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7fedf7583d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-03-20 22:54:57.192899: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
97it [00:49,  2.47it/s]2025-03-20 22:55:46.655638: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
97it [00:49,  1.96it/s]
2025-03-20 22:56:28.847075: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [2]:
num_trajs

92